In [8]:
from google.colab import files
uploaded = files.upload()

Saving black_flower.jpeg to black_flower (1).jpeg


In [17]:
from PIL import Image, ImageEnhance, ImageFilter, ImageOps, ImageDraw
import random, math

input_path = list(uploaded.keys())[0]
output_path = "output_future_final.jpg"

# ──────────────────────────────────────────────
# 1. CYBERPUNK COLOR GRADE
# Crushes greens, boosts cyan + magenta tones
# ──────────────────────────────────────────────
def apply_cyberpunk_grade(image):
    pixels = image.load()
    result = Image.new("RGB", image.size)
    res_px = result.load()
    for y in range(image.height):
        for x in range(image.width):
            r, g, b = pixels[x, y]

            # Push reds → magenta, blues → cyan, crush mid greens
            r = min(int(r * 1.15 + 20), 255)       # lift reds → warm neon
            g = min(int(g * 0.60), 255)             # crush greens hard
            b = min(int(b * 1.35 + 30), 255)        # push blues → electric

            # Teal shadow tint in dark areas
            brightness = (r + g + b) // 3
            if brightness < 80:
                r = min(max(r - 10, 0), 255)
                g = min(max(g + 25, 0), 255)
                b = min(max(b + 40, 0), 255)

            # Magenta highlight tint in bright areas
            if brightness > 180:
                r = min(r + 30, 255)
                g = min(max(g - 15, 0), 255)
                b = min(b + 20, 255)

            res_px[x, y] = (
                min(max(r, 0), 255),
                min(max(g, 0), 255),
                min(max(b, 0), 255)
            )
    return result

# ──────────────────────────────────────────────
# 2. CHROMATIC ABERRATION
# Splits RGB channels and offsets them
# ──────────────────────────────────────────────
def apply_chromatic_aberration(image, shift=6):
    r, g, b = image.split()

    # Shift red channel left, blue channel right
    def shift_channel(channel, dx, dy):
        shifted = Image.new("L", channel.size, 0)
        px_src = channel.load()
        px_dst = shifted.load()
        w, h = channel.size
        for y in range(h):
            for x in range(w):
                nx, ny = x + dx, y + dy
                if 0 <= nx < w and 0 <= ny < h:
                    px_dst[nx, ny] = px_src[x, y]
        return shifted

    r_shifted = shift_channel(r, -shift, 0)
    b_shifted = shift_channel(b,  shift, 1)

    return Image.merge("RGB", (r_shifted, g, b_shifted))

# ──────────────────────────────────────────────
# 3. SCANLINES
# Simulates CRT / holographic display lines
# ──────────────────────────────────────────────
def apply_scanlines(image, gap=3, darkness=0.55):
    result = image.copy()
    pixels = result.load()
    for y in range(image.height):
        if y % gap == 0:
            for x in range(image.width):
                r, g, b = pixels[x, y]
                pixels[x, y] = (
                    int(r * darkness),
                    int(g * darkness),
                    int(b * darkness)
                )
    return result

# ──────────────────────────────────────────────
# 4. GLITCH EFFECT
# Randomly shifts horizontal pixel rows
# ──────────────────────────────────────────────
def apply_glitch(image, num_glitches=18, max_shift=22):
    result = image.copy()
    pixels = result.load()
    src_pixels = image.load()
    w, h = image.size

    for _ in range(num_glitches):
        y_start = random.randint(0, h - 1)
        band_height = random.randint(1, 6)
        shift = random.randint(-max_shift, max_shift)
        tint = random.choice([
            (30, 0, 0), (0, 0, 30), (0, 30, 30),   # red/blue/cyan glitch
            (-20, -20, -20), (20, 20, 20)
        ])

        for dy in range(band_height):
            y = min(y_start + dy, h - 1)
            for x in range(w):
                src_x = (x + shift) % w
                r, g, b = src_pixels[src_x, y]
                pixels[x, y] = (
                    min(max(r + tint[0], 0), 255),
                    min(max(g + tint[1], 0), 255),
                    min(max(b + tint[2], 0), 255),
                )
    return result

# ──────────────────────────────────────────────
# 5. NEON GLOW
# Brightens highlights with colored neon bloom
# ──────────────────────────────────────────────
def apply_neon_glow(image, threshold=170, glow_color=(0, 255, 220), strength=0.45):
    glow_layer = Image.new("RGB", image.size, (0, 0, 0))
    glow_px = glow_layer.load()
    src_px = image.load()

    for y in range(image.height):
        for x in range(image.width):
            r, g, b = src_px[x, y]
            brightness = (r + g + b) // 3
            if brightness > threshold:
                factor = (brightness - threshold) / (255 - threshold)
                glow_px[x, y] = (
                    int(glow_color[0] * factor),
                    int(glow_color[1] * factor),
                    int(glow_color[2] * factor),
                )

    # Blur glow layer for bloom spread
    glow_layer = glow_layer.filter(ImageFilter.GaussianBlur(radius=8))

    # Blend glow back onto image
    result = Image.new("RGB", image.size)
    res_px = result.load()
    img_px = image.load()
    gl_px = glow_layer.load()

    for y in range(image.height):
        for x in range(image.width):
            r = min(img_px[x, y][0] + int(gl_px[x, y][0] * strength), 255)
            g = min(img_px[x, y][1] + int(gl_px[x, y][1] * strength), 255)
            b = min(img_px[x, y][2] + int(gl_px[x, y][2] * strength), 255)
            res_px[x, y] = (r, g, b)
    return result

# ──────────────────────────────────────────────
# 6. HUD GRID OVERLAY
# Draws a futuristic targeting grid + corner brackets
# ──────────────────────────────────────────────
def apply_hud_grid(image, grid_spacing=60, grid_color=(0, 255, 200, 35),
                   bracket_color=(0, 255, 200, 180)):
    overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    w, h = image.size

    # Draw grid lines
    for x in range(0, w, grid_spacing):
        draw.line([(x, 0), (x, h)], fill=grid_color, width=1)
    for y in range(0, h, grid_spacing):
        draw.line([(0, y), (w, y)], fill=grid_color, width=1)

    # Corner bracket marks (top-left, top-right, bottom-left, bottom-right)
    bracket_len = 40
    bracket_thickness = 3
    margin = 30
    corners = [
        (margin, margin),
        (w - margin, margin),
        (margin, h - margin),
        (w - margin, h - margin),
    ]
    directions = [
        (1, 1), (-1, 1), (1, -1), (-1, -1)
    ]
    for (cx, cy), (dx, dy) in zip(corners, directions):
        # Horizontal arm
        draw.line([(cx, cy), (cx + dx * bracket_len, cy)],
                  fill=bracket_color, width=bracket_thickness)
        # Vertical arm
        draw.line([(cx, cy), (cx, cy + dy * bracket_len)],
                  fill=bracket_color, width=bracket_thickness)

    # Center crosshair
    cx, cy = w // 2, h // 2
    cross_size = 18
    gap = 6
    draw.line([(cx - cross_size, cy), (cx - gap, cy)], fill=bracket_color, width=2)
    draw.line([(cx + gap, cy), (cx + cross_size, cy)], fill=bracket_color, width=2)
    draw.line([(cx, cy - cross_size), (cx, cy - gap)], fill=bracket_color, width=2)
    draw.line([(cx, cy + gap), (cx, cy + cross_size)], fill=bracket_color, width=2)

    # Blend overlay
    base = image.convert("RGBA")
    composite = Image.alpha_composite(base, overlay)
    return composite.convert("RGB")

# ──────────────────────────────────────────────
# 7. VIGNETTE (dark edge glow like a screen)
# ──────────────────────────────────────────────
def apply_future_vignette(image, tint=(0, 10, 30)):
    cx, cy = image.width / 2, image.height / 2
    max_dist = math.sqrt(cx**2 + cy**2)
    result = image.copy()
    pixels = result.load()

    for y in range(image.height):
        for x in range(image.width):
            dist = math.sqrt((x - cx)**2 + (y - cy)**2)
            falloff = (dist / max_dist) ** 2.2
            r, g, b = pixels[x, y]
            # Darken + push toward tint color at edges
            pixels[x, y] = (
                int(r * (1 - falloff) + tint[0] * falloff),
                int(g * (1 - falloff) + tint[1] * falloff),
                int(b * (1 - falloff) + tint[2] * falloff),
            )
    return result

# ──────────────────────────────────────────────
# MASTER PIPELINE
# ──────────────────────────────────────────────
def future_effect(input_path, output_path):
    image = Image.open(input_path).convert("RGB")

    print("🔵 Applying cyberpunk color grade...")
    image = apply_cyberpunk_grade(image)

    print("🔵 Boosting contrast for neon pop...")
    image = ImageEnhance.Contrast(image).enhance(1.45)
    image = ImageEnhance.Sharpness(image).enhance(2.0)

    print("🔵 Applying chromatic aberration...")
    image = apply_chromatic_aberration(image, shift=7)

    print("🔵 Applying neon glow bloom...")
    image = apply_neon_glow(image, threshold=160,
                            glow_color=(0, 255, 220), strength=0.50)

    print("🔵 Adding glitch bands...")
    image = apply_glitch(image, num_glitches=20, max_shift=25)

    print("🔵 Overlaying scanlines...")
    image = apply_scanlines(image, gap=3, darkness=0.50)

    print("🔵 Applying HUD grid overlay...")
    image = apply_hud_grid(image, grid_spacing=55)

    print("🔵 Adding screen vignette...")
    image = apply_future_vignette(image, tint=(0, 8, 25))

    image.save(output_path, quality=95)
    print(f"✅ Saved → {output_path}")
    return image

result = future_effect(input_path, output_path)

🔵 Applying cyberpunk color grade...
🔵 Boosting contrast for neon pop...
🔵 Applying chromatic aberration...
🔵 Applying neon glow bloom...
🔵 Adding glitch bands...
🔵 Overlaying scanlines...
🔵 Applying HUD grid overlay...
🔵 Adding screen vignette...
✅ Saved → output_future.jpg
